results of each tracking are in video files

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
import tensorflow as tf
import time
import os
import json
import kagglehub
from djitellopy import Tello
from datetime import datetime
from ultralytics import YOLO
from collections import defaultdict

path = kagglehub.model_download("tensorflow/ssd-mobilenet-v2/tensorFlow2/fpnlite-320x320")

In [ ]:
# Models
modelCamera = tf.saved_model.load("saved_model")
modelDrone = YOLO("YOLO/yolov8s.pt")

# Function to perform human detection and distance estimation
def detect_and_estimate_distance(frame, threshold_area, model=modelCamera):
    input_tensor = tf.convert_to_tensor(frame)
    input_tensor = input_tensor[tf.newaxis,...]
    
    detections = model(input_tensor)

    boxes = detections['detection_boxes'][0].numpy()
    classes = detections['detection_classes'][0].numpy().astype(np.int32)
    scores = detections['detection_scores'][0].numpy()

    frame_height, frame_width, _ = frame.shape

    for i in range(len(scores)):
        if scores[i] > 0.5 and classes[i] == 1:  # Class 1 corresponds to 'person'
            ymin, xmin, ymax, xmax = boxes[i]
            (left, right, top, bottom) = (xmin * frame_width, xmax * frame_width, ymin * frame_height, ymax * frame_height)
            area = (right - left) * (bottom - top)
            
            # Draw bounding box
            cv2.rectangle(frame, (int(left), int(top)), (int(right), int(bottom)), (0, 255, 0), 2)
            
            # Check if area exceeds threshold
            if area > threshold_area:
                return True, frame
    
    return False, frame


def detect_objects_in_video(video_path, output_video_path, model=modelDrone):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    object_counts = defaultdict(int)

    # Get frame width and height
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))

    # Define the codec and create VideoWriter object for the output video
    out = cv2.VideoWriter(f'{output_video_path}/drone_with_detection.avi', cv2.VideoWriter_fourcc('M','J','P','G'), 30.0, (frame_width, frame_height))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detect objects in the frame
        results = model(frame)

        cv2.imshow("Frame", frame)

        for result in results:
            for box in result.boxes:
                class_id = int(box.cls[0])
                label = model.names[class_id]
                object_counts[label] += 1

                # Draw bounding boxes and labels on the frame
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                confidence = box.conf[0]
                color = (0, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"{label} {confidence:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Write the processed frame to the output video
        out.write(frame)
        frame_count += 1

    cap.release()
    out.release()
    return object_counts, frame_count

def calculate_percentage(object_counts, frame_count):
    percentages = {obj: (count / frame_count) * 100 for obj, count in object_counts.items()}
    return percentages

def save_json(data, output_folder):
    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)
    
   # Sort data by percentages in descending order
    sorted_data = dict(sorted(data.items(), key=lambda item: item[1], reverse=True))

    # Define the output JSON file path
    json_file_path = os.path.join(output_folder, "object_detection_results.json")

    # Save sorted data as JSON
    with open(json_file_path, "w") as json_file:
        json.dump(sorted_data, json_file, indent=4)


def run_video_object_detection(video_path, output_folder, model=modelDrone):

    # Perform object detection and count objects
    object_counts, frame_count = detect_objects_in_video(video_path, output_folder, model)
    
    # Calculate percentages
    object_percentages = calculate_percentage(object_counts, frame_count)

    # Prepare data for JSON
    data = {obj: f"{percentage:.2f}%" for obj, percentage in object_percentages.items()}

    # Save the results to a JSON file
    save_json(data, output_folder)
    
    print(f"Object detection results and video saved to {output_folder}/")


def list_connected_devices(max_devices=10):
    available_devices = []
    for device_index in range(max_devices):
        cap = cv2.VideoCapture(device_index)
        if cap.isOpened():
            available_devices.append(device_index)
            cap.release()
    return available_devices
#print("Connected video devices:", list_connected_devices())

def tello_fly_record():

    # Create a Tello object
    tello = Tello()

    # Connect to the Tello drone
    tello.connect()

    # Print the battery level
    battery = tello.get_battery()
    print(f"Battery level: {battery}%")

    # Get current date for folder name
    current_date = datetime.now().strftime("%Y-%m-%d")
    if not os.path.exists(f'{current_date}'):
        os.makedirs(f'{current_date}')

    # Start video stream
    tello.streamon()

    # Initialize the video writer variable
    frame_read = tello.get_frame_read()
    time.sleep(2)
    frame = frame_read.frame
    height, width, _ = frame.shape

    video = None  # Initialize video writer variable
    
    print("Starting video capture. Press 'q' to quit.")

    # Take off
    tello.takeoff()

    # Initialize the video writer once the drone takes off
    video = cv2.VideoWriter(f'{current_date}/drone.avi', cv2.VideoWriter_fourcc('M','J','P','G'), 30.0, (width, height))

    try:
        # Move up by 1.5 meter
        tello.move_up(150)
        time.sleep(1)

        # Move forward by 1 meter
        tello.move_forward(100)
        time.sleep(1)

        # Move back by 1 meter
        tello.move_back(100)
        time.sleep(1)

    except Exception as e:
        print(f"An error occurred during drone movement: {e}")
        

    while tello.get_height() > 0:  # Continue recording while the drone is airborne
        # Get the current frame from the drone
        frame = frame_read.frame
        
        # Write the frame to the video file
        if video:
            video.write(frame)
        
        # Display the frame (optional)
        cv2.imshow("Tello Video Stream", frame)
        
        # Check for 'q' key press to quit early
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Stop recording when the drone lands
    if video:
        video.release()  
        cv2.destroyAllWindows() 

    # Land the drone
    tello.land()

    # Stop video stream
    tello.streamoff()

    # Disconnect from the drone (optional, depending on your Tello library version)
    #tello.end()

    # Perform object detection on the recorded video
    run_video_object_detection(f'{current_date}/drone.avi', current_date, model=modelDrone)

def main(camera):
    # Access webcam and process frames
    cap = cv2.VideoCapture(camera)
    threshold_area = 50000  # Define an appropriate threshold area

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        is_close, processed_frame = detect_and_estimate_distance(frame, threshold_area, model=modelCamera)

        cv2.imshow("Frame", processed_frame)

        if is_close:
            # Run drone.py
            # Save frame to folder with name of current date
            current_date = datetime.now().strftime("%Y-%m-%d")
            if not os.path.exists(f'{current_date}'):
                os.makedirs(f'{current_date}')
            frame_path = os.path.join(f'{current_date}', f"frame.jpg")
            cv2.imwrite(frame_path, processed_frame)

            # Initialize the drone
            tello_fly_record()

            break
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Run the main function
main(0)

#if __name__ == "__main__":
    #main()

In [41]:
import requests
import json
import os
import base64
from datetime import datetime

def encode_credentials(username, password):
    credentials = f"{username}:{password}"
    return base64.b64encode(credentials.encode('ascii')).decode('ascii')

def make_sort_percentage_in_json():
    current_date = datetime.now().strftime("%Y-%m-%d")
    with open(f'{current_date}/object_detection_results.json', 'r') as file:
        data = json.load(file)
    sorted_data = {k: v for k, v in sorted(data.items(), key=lambda item: float(item[1].strip('%')), reverse=True)}
    return sorted_data

def upload_media(file_path, media_type, wordpress_url, auth_header):
    file_name = os.path.basename(file_path)
    with open(file_path, 'rb') as file:
        response = requests.post(
            f'{wordpress_url}/wp-json/wp/v2/media',
            headers={
                'Authorization': auth_header,
                'Content-Disposition': f'attachment; filename={file_name}'
            },
            files={'file': (file_name, file, media_type)}
        )
        print(f"Media upload status code: {response.status_code}")
        print(f"Media upload response: {response.text}")
        response.raise_for_status()
        media_id = response.json()['id']
        media_link = response.json()['source_url']
        return media_id, media_link

def create_post(post_data, wordpress_url, auth_header):
    response = requests.post(
        f'{wordpress_url}/wp-json/wp/v2/posts',
        headers={
            'Authorization': auth_header,
            'Content-Type': 'application/json'
        },
        json=post_data
    )
    print(f"Post creation status code: {response.status_code}")
    print(f"Post creation response: {response.text}")
    response.raise_for_status()
    return response.json()

def make_wp_post():
    wordpress_url = "https://demos.greenshiftwp.com/wordpress-python"
    consumer_key = "fleeke@gmail.com"
    consumer_secret = "xI1f 521b xN9K Cf0b EDMF UkkT"
    auth_header = f"Basic {encode_credentials(consumer_key, consumer_secret)}"

    current_date = datetime.now().strftime("%Y-%m-%d")
    video_path_detected = f'{current_date}/drone_with_detection.avi'
    video_path = f'{current_date}/drone.avi'
    image_path = f'{current_date}/frame.jpg'

    video_id, video_link = upload_media(video_path, "video/mp4", wordpress_url, auth_header)
    video_id_detected, video_link_detected = upload_media(video_path_detected, "video/mp4", wordpress_url, auth_header)
    image_id, _ = upload_media(image_path, "image/jpeg", wordpress_url, auth_header)

    json_detected = make_sort_percentage_in_json()
    json_detected_str = json.dumps(json_detected)

    post_data = {
        "title": f"Intruder Detected on {current_date}",
        "content": "",
        "status": "publish",
        "featured_media": image_id,
        "meta": {
            "video": video_link,
            "video_detected": video_link_detected,
            "json": json_detected_str
        }
    }

    post = create_post(post_data, wordpress_url, auth_header)
    print(f"New post created: {post['link']}")

make_wp_post()

Media upload status code: 201
Media upload response: {"id":12,"date":"2024-08-10T17:31:51","date_gmt":"2024-08-10T17:31:51","guid":{"rendered":"https:\/\/demos.greenshiftwp.com\/wordpress-python\/wp-content\/uploads\/sites\/8\/2024\/08\/drone.avi","raw":"https:\/\/demos.greenshiftwp.com\/wordpress-python\/wp-content\/uploads\/sites\/8\/2024\/08\/drone.avi"},"modified":"2024-08-10T17:31:51","modified_gmt":"2024-08-10T17:31:51","slug":"drone","status":"inherit","type":"attachment","link":"https:\/\/demos.greenshiftwp.com\/wordpress-python\/drone\/","title":{"raw":"drone","rendered":"drone"},"author":1,"featured_media":0,"comment_status":"open","ping_status":"closed","template":"","meta":{"_gspb_post_css":"","demourl":"","download_url":"","download_url_animated":""},"permalink_template":"https:\/\/demos.greenshiftwp.com\/wordpress-python\/?attachment_id=12","generated_slug":"drone","class_list":["post-12","attachment","type-attachment","status-inherit","hentry"],"description":{"raw":"","r